# Solutions 11: Evaluation, Failure Modes, and Security

This notebook solves the four exercises of Lab 11. Execution status: exercises 1 and 2 run
fully live (the n-gram sweep on the real corpus, and the `LengthMonitor` with its unit
tests); exercises 3 and 4 gate their training and retraining runs behind `RUN_EVAL = False`
and instead run live the part a defender can always run: the accounting. Exercise 4's bias
measurement on real logits also runs live. Attempt the exercises yourself before reading this
file.

The shared idea across all four solutions is the lab's own: a detector, a monitor, or a
defense is only trusted after it has fired correctly on a case with a known answer, and a
cost claim is only trusted after its arithmetic has been recomputed two ways.

In [1]:
import sys, os, json, math
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from kd_core import topk_truncation_bias, bytes_per_token_cache
from kd_pipeline import set_seed_everywhere, EntropyMonitor, \
                        bandwidth_bound_decode_tps, decode_wallclock_hours

RUN_EVAL = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)
print(f"torch {torch.__version__} | RUN_EVAL: {RUN_EVAL}")

torch 2.13.0+cpu | RUN_EVAL: False


## Exercise 1: Contamination sensitivity, the n-gram size sweep

**The exercise, restated.** Sweep the n-gram size from 4 to 12 on the Part A-1 contamination
checker. Where does chance overlap die out on this corpus? That knee, the point where the
flag rate stops falling, justifies or corrects the lab's choice of 8.

**The approach.** This runs fully live on the same data the lab's checker ran on: content
tokens only (the completion tokens under the mask, because Part A-1 showed that raw sequences
make the checker measure the chat template instead of the data), 512 train rows against 128
eval rows, threshold 0.3, everything held fixed except n. Two curves answer the question.
The flagged fraction per n is the operational curve: it says how many eval rows the checker
would discard at each n. The mean best-overlap score per n is the mechanistic curve: it is
the average, over eval rows, of the highest overlap fraction any train row achieves, and it
shows chance overlap directly, because in a corpus with no true duplicates this number is
made entirely of coincidences (stock phrases, common instruction wording) whose probability
of colliding falls fast as the required run of matching tokens grows. The knee is read off
the flag curve as the smallest n whose flag count has already reached the level it holds
through n = 12; the asserts check that the mean-overlap curve falls monotonically and has
collapsed by n = 12, which is what "chance overlap dies" means in numbers.

In [2]:
tr = torch.load("../data/lab03/train.pt"); ev = torch.load("../data/lab03/eval.pt")
content = lambda ids, mask: [int(t) for t, mk in zip(ids.tolist(), mask.tolist()) if mk]
train_rows = [content(tr["input_ids"][i], tr["mask"][i]) for i in range(512)]
eval_rows  = [content(ev["input_ids"][i], ev["mask"][i]) for i in range(128)]

def ngrams(ids, n):
    return {tuple(ids[i:i+n]) for i in range(len(ids) - n + 1)}

NS = list(range(4, 13))
flag_frac, mean_overlap, flagged_ids = [], [], {}
for n in NS:
    train_grams = [ngrams(r, n) for r in train_rows]
    scores = []
    for er in eval_rows:
        eg = ngrams(er, n)
        scores.append(max((len(eg & tg) / max(1, len(eg)) for tg in train_grams),
                          default=0.0))
    fl = [i for i, s in enumerate(scores) if s > 0.3]
    flag_frac.append(len(fl) / len(eval_rows))
    mean_overlap.append(sum(scores) / len(scores))
    flagged_ids[n] = fl
    print(f"n={n:>2}: flagged {len(fl):>2}/128  mean best-overlap {mean_overlap[-1]:.4f}")

floor_flags = flag_frac[-1]                       # the level held at n=12
knee = next(n for n, f in zip(NS, flag_frac) if f == floor_flags)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(NS, flag_frac, "o-", lw=2, label="flagged fraction (threshold 0.3)")
ax.plot(NS, mean_overlap, "s--", lw=2, label="mean best-overlap score")
ax.axvline(8, color="#888", ls=":", lw=1); ax.text(8.05, max(flag_frac)*0.9, "lab's choice", fontsize=8)
ax.axvline(knee, color="#c33", ls="--", lw=1); ax.text(knee+0.05, max(flag_frac)*0.75, "knee", color="#c33", fontsize=8)
ax.set_xlabel("n-gram size"); ax.set_ylabel("fraction"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("../figures/sol11_ngram_sweep.png", dpi=120)
print(f"knee (flag rate reaches its floor): n={knee} | sweep plot saved")

assert all(a >= b for a, b in zip(mean_overlap, mean_overlap[1:])), \
    "chance overlap must fall monotonically as n grows"
assert mean_overlap[-1] < 0.25 * mean_overlap[0], \
    "by n=12 chance overlap must have collapsed relative to n=4"
assert all(a >= b for a, b in zip(flag_frac, flag_frac[1:])), \
    "the flag rate must never rise as the evidence bar rises"
assert knee <= 8, "the lab's 8-gram choice must sit at or past the knee"
print(f"rows flagged only at short n: {sorted(set(flagged_ids[4]) - set(flagged_ids[8]))} "
      f"(stock-phrase collisions, not contamination)")


n= 4: flagged  1/128  mean best-overlap 0.0818
n= 5: flagged  1/128  mean best-overlap 0.0603


n= 6: flagged  0/128  mean best-overlap 0.0469
n= 7: flagged  0/128  mean best-overlap 0.0359


n= 8: flagged  0/128  mean best-overlap 0.0276
n= 9: flagged  0/128  mean best-overlap 0.0213


n=10: flagged  0/128  mean best-overlap 0.0159
n=11: flagged  0/128  mean best-overlap 0.0119


n=12: flagged  0/128  mean best-overlap 0.0088
knee (flag rate reaches its floor): n=6 | sweep plot saved
rows flagged only at short n: [6] (stock-phrase collisions, not contamination)


**Interpretation.** The printed sweep locates the knee at n = 6 on this corpus: one
eval row (row 6) is flagged at n = 4 and n = 5, and from n = 6 through n = 12 the flag count
sits at zero and stays there. That short-n flag is chance overlap caught in the act, a row
sharing 4-token and 5-token runs with some train row (stock instruction phrasing collides at
that length easily) but no longer sharing anything once six consecutive matching tokens are
required. The mean best-overlap curve tells the same story continuously: it falls
monotonically from about 0.082 at n = 4 to about 0.009 at n = 12, a nine-fold collapse, and
both monotonicity asserts passed.

So the lab's 8-gram choice is justified rather than corrected, and the sweep says something
more useful than "8 was fine": it prices the margin. The knee at 6 means 8 carries two
n-gram sizes of safety against corpus-specific stock phrasing, while the still-falling mean
curve past 8 shows what raising n further buys (less chance overlap) and costs (a true
near-duplicate with small edits shares fewer long n-grams than short ones, so sensitivity to
real contamination falls as n rises; the planted-duplicate test in Part A-1 is the guard on
that side). On a different corpus with longer boilerplate, template-heavy code for example,
the knee moves right, and this sweep is the ten-second measurement that finds it before the
threshold gets copied blindly.

## Exercise 2: The length-collapse tripwire

**The exercise, restated.** Extend `EntropyMonitor` into a `LengthMonitor` with the same
windowed-drop rule applied to mean generation length, unit-test it against gallery shape 3
(length collapse), and retrofit it into Lab 07's callback.

**The approach.** The monitor is a direct structural copy of `EntropyMonitor`, because the
lab's Part A-2 established that the windowed-drop rule (fire when the tracked value falls
below an absolute floor, or loses more than a set fraction of its value within a trailing
window of observations) is the right shape for collapse detection generally, and the gallery
showed that length collapse is the same trajectory shape as entropy collapse on a different
axis. The unit tests follow the lab's testing discipline for detectors: the monitor must fire
on the known-bad case, stay quiet on the known-good cases, and, since the class has two
firing rules, each rule is exercised separately. Gallery shape 3 (length decaying from 85
toward 30) tests the floor rule. A synthetic abrupt drop that stays above the floor (90 down
to 54 within one window) tests the windowed rule specifically, and the test asserts the floor
was *not* the trigger, so a bug that disabled the windowed rule could not pass by accident.
The two healthy controls are the gallery's healthy shape (length slowly rising to 90) and the
entropy-collapse shape's constant length of 80, which is the near-enemy: a run where entropy
is collapsing but length is not, and a length monitor has no business firing.

In [3]:
class LengthMonitor:
    # EntropyMonitor's windowed-drop rule, retargeted at mean generation length.
    def __init__(self, floor_len=45.0, drop_frac=0.35, window=8):
        self.floor_len, self.drop_frac, self.window = floor_len, drop_frac, window
        self.history = []

    def update(self, step, mean_len):
        self.history.append((step, float(mean_len)))

    @property
    def collapsed(self):
        if len(self.history) < 3:
            return False
        latest = self.history[-1][1]
        if latest < self.floor_len:
            return True
        recent = [h for _, h in self.history[-self.window:]]
        return latest < recent[0] * (1 - self.drop_frac)

    def report(self):
        if not self.history:
            return "no observations"
        (s0, l0), (s1, l1) = self.history[0], self.history[-1]
        return (f"length {l0:.0f} @ step {s0} -> {l1:.0f} @ step {s1} "
                f"({'COLLAPSED' if self.collapsed else 'healthy'})")

steps = torch.arange(0, 2000, 25).float()          # the gallery's exact time axis
shapes = {
    "healthy":            90 - 10 * torch.exp(-steps / 500),
    "entropy collapse":   80 * torch.ones_like(steps),      # near-enemy: length is fine
    "length collapse":    30 + 55 * torch.exp(-steps / 400),  # gallery shape 3
}
first_fire = {}
for name, lens in shapes.items():
    mon = LengthMonitor()
    fired_at = None
    for s, l in zip(steps.tolist(), lens.tolist()):
        mon.update(int(s), l)
        if fired_at is None and mon.collapsed:
            fired_at = int(s)
    first_fire[name] = fired_at
    verdict = mon.collapsed
    assert verdict == (name == "length collapse"), (name, verdict)
    print(f"{name:<18}: {mon.report()}" +
          (f" | first fired at step {fired_at}" if fired_at else ""))

# The windowed rule on its own: an abrupt halving that never crosses the floor.
mon = LengthMonitor()
for i, l in enumerate([90] * 36 + [78, 66, 54]):
    mon.update(i * 25, l)
assert mon.collapsed and mon.history[-1][1] > mon.floor_len, \
    "the windowed rule must fire on a fast drop even while the floor rule is silent"
print(f"abrupt-drop test: fired at length {mon.history[-1][1]:.0f}, "
      f"above the floor of {mon.floor_len:.0f} -> the windowed rule fired, not the floor")

if RUN_EVAL:
    # The retrofit into Lab 07's on-policy callback is the same three lines the
    # EntropyMonitor already occupies there:
    #   length_mon = LengthMonitor()                                  # next to EntropyMonitor
    #   length_mon.update(state.global_step, float(gen_mask.sum(1).float().mean()))
    #   if length_mon.collapsed: control.should_training_stop = True  # same tripwire wiring
    pass
else:
    print("RUN_EVAL=False: the Lab 07 retrofit is gated; the monitor itself is proven above")

healthy           : length 80 @ step 0 -> 90 @ step 1975 (healthy)
entropy collapse  : length 80 @ step 0 -> 80 @ step 1975 (healthy)
length collapse   : length 85 @ step 0 -> 30 @ step 1975 (COLLAPSED) | first fired at step 525
abrupt-drop test: fired at length 54, above the floor of 45 -> the windowed rule fired, not the floor
RUN_EVAL=False: the Lab 07 retrofit is gated; the monitor itself is proven above


**Interpretation.** All four tests passed, and each one retires a specific bug class.
The gallery shape-3 test shows the monitor firing at step 525, which is worth translating
into operational terms: the gallery's length collapse runs 2,000 steps, so this tripwire cuts
the loss at about a quarter of the way in, saving three quarters of a doomed run's budget,
where the lab's motivating observation was that a human watching per-token metrics notices
length collapse "far too late" because every per-token number stays good throughout. The
constant-length near-enemy staying quiet matters just as much: entropy collapse and length
collapse need different responses (Lab 07's discussion versus early stopping and loss-term
inspection), so a monitor that conflated them would misdirect the debugging. And the
abrupt-drop test proves the windowed rule works independently of the floor, with the printed
line showing the firing length of 54 sitting above the floor of 45.

The floor and fraction values (45 tokens, 35% within 8 observations) are tuned to the
gallery's scale, and the honest caveat from the lab's `EntropyMonitor` docstring carries
over verbatim: tune both on your own runs; the point is having *an* automatic tripwire. The
gated retrofit is three lines because the monitor deliberately shares `EntropyMonitor`'s
interface; on the training box it drops into Lab 07's callback next to its sibling, and the
pair covers the two collapse axes the failure gallery says on-policy runs actually die
on.

## Exercise 3: The extraction curve

**The exercise, restated.** Rerun Lab 06's SeqKD arm at corpus sizes of 256, 1,024, and 4,096
prompts and plot student quality against queries. That plot is your model's queries-to-clone
curve seen from the attacker's chair. Where would a per-key rate limit actually bind?

**The approach.** The three training runs are gated. What runs live is the accounting that
turns Lab 06's cost table into the attacker's planning document, because the exercise's
insight is that they are the same table read from a different chair. The quantities: each
query is one prompt generating 256 tokens (Lab 06's corpus recipe), so a corpus of Q prompts
is Q queries and Q x 256 generated tokens. The defender's serving cost per query comes from
the course's bandwidth arithmetic: a 1.7B teacher in bf16 (2 bytes per parameter) on hardware
with 273 GB/s of memory bandwidth decodes at most 273 / (1.7 x 2) = 80 tokens per second per
stream, because every generated token must stream every weight through the compute units
once. The attacker-side constraint is a per-key rate limit, taken here as 1,000 queries per
day per key, a realistic free-tier order of magnitude. The table computes, per corpus size:
queries, tokens, the defender's single-stream serving hours, and the days one key needs. The
assert is internal consistency, computed two independent ways where possible: token counts
from the corpus recipe, hours from the course's own `decode_wallclock_hours` against a manual
recomputation, and the 16x ratio that must hold between the largest and smallest corpus
because the accounting is linear in Q.

In [4]:
TEACHER_B, BW_GBS, NEW_TOKS = 1.7, 273.0, 256
RATE_LIMIT_PER_DAY = 1000
tps = bandwidth_bound_decode_tps(TEACHER_B, BW_GBS)          # 273/(1.7*2) = 80.3 tok/s

print(f"teacher {TEACHER_B}B @ {BW_GBS:.0f} GB/s -> roofline {tps:.1f} tok/s per stream\n")
print(f"{'prompts':>8} {'queries':>8} {'tokens':>10} {'serve h (1 stream)':>19} "
      f"{'days @ 1k/key/day':>18}")
rows = {}
for Q in (256, 1024, 4096):
    tokens = Q * NEW_TOKS
    hours = decode_wallclock_hours(tokens, tps)
    days_one_key = Q / RATE_LIMIT_PER_DAY
    rows[Q] = dict(tokens=tokens, hours=hours, days=days_one_key)
    print(f"{Q:>8} {Q:>8} {tokens:>10,} {hours:>19.2f} {days_one_key:>18.2f}")

# Internal consistency, recomputed independently of the library calls.
for Q, r in rows.items():
    assert r["tokens"] == Q * NEW_TOKS, "token accounting must follow the corpus recipe"
    manual_hours = r["tokens"] * (TEACHER_B * 2.0 / BW_GBS) / 3600.0
    assert abs(manual_hours - r["hours"]) < 1e-9, "two derivations of hours must agree"
assert abs(rows[4096]["hours"] / rows[256]["hours"] - 16.0) < 1e-9, \
    "linear accounting: 16x the queries is 16x the cost, on both sides of the API"
binding = [Q for Q, r in rows.items() if r["days"] > 1.0]
print(f"\ncorpus sizes where a 1k/day key needs multiple days: {binding}")
print("consistency asserts passed: the attacker's bill and the defender's bill are the "
      "same number viewed from opposite chairs")

if RUN_EVAL:
    # The gated quality column: three Lab 06 SeqKD runs, identical except the
    # corpus size, each scored by this lab's decontaminated harness; plot
    # quality vs queries and read off queries-to-X%.
    pass
else:
    print("RUN_EVAL=False: the quality column of the extraction curve is gated")

teacher 1.7B @ 273 GB/s -> roofline 80.3 tok/s per stream

 prompts  queries     tokens  serve h (1 stream)  days @ 1k/key/day
     256      256     65,536                0.23               0.26
    1024     1024    262,144                0.91               1.02
    4096     4096  1,048,576                3.63               4.10

corpus sizes where a 1k/day key needs multiple days: [1024, 4096]
consistency asserts passed: the attacker's bill and the defender's bill are the same number viewed from opposite chairs
RUN_EVAL=False: the quality column of the extraction curve is gated


**Interpretation.** The live table is the extraction curve's x-axis and price sheet,
verified: 256 prompts is 65,536 generated tokens and about 0.23 single-stream serving hours,
scaling linearly (the asserted 16x) to 4,096 prompts, about a million tokens, and 3.6 hours,
with batching dividing the defender's wall-clock by an order of magnitude as Lab 06 priced.
The printed binding line answers the exercise's final question: a 1,000-query-per-day key
does not bind at all at 256 prompts, barely grazes the limit at 1,024 (1.02 days, so one
key and a patient overnight run), and binds meaningfully only at 4,096, where one key needs
about four days or, equivalently, four keys need one day. Since Lab 06's Part C expects the
SeqKD quality curve to show strong gains already at small corpus sizes with diminishing
returns after, the uncomfortable conclusion is that the rate limit binds exactly where the
attacker no longer needs volume: the cheap early part of the curve, where most of the clone
quality is bought, fits under any realistic per-key limit.

The gated runs fill in the quality column; the expected shape, grounded in Lab 06's Part C
ordering, is steep quality gain from 256 to 1,024 and a flattening toward 4,096, and no
number in this solution pretends to know those values before the runs execute. What the live
accounting already establishes is the defender's real lever: per-key limits price *volume*,
but the curve's shape means the defense that matters must price *the first thousand queries*,
which is why the exercise pairs with exercise 4, where the defender degrades what each query
returns rather than how many are allowed.

## Exercise 4: Defense pricing

**The exercise, restated.** Recompute Lab 04's cache at k in {1, 5} and retrain: this is
logit truncation (serving only the top-k token probabilities instead of all of them) as an
extraction defense, priced in student quality. Combine with exercise 3 for the defender's
trade-off chart.

**The approach.** The two retraining runs are gated. What runs live is the measurement that
predicts their outcome, using Lab 02's own instrument on real logits: `topk_truncation_bias`
on the course's standard pair (360M as teacher, 135M as student, real eval rows), at the
defense's k values plus k = 64 as the reference the course's caches actually use. Two
columns of that table price the defense before any retraining. The mean mass covered says
how much of the teacher's probability mass a k-limited API still hands over, which is the
information the defense fails to withhold. The truncation bias of the two KL estimators
(renormalising over the kept entries versus keeping an explicit tail bucket) says how
distorted the distillation signal becomes at that k, which is what the attacker's student
would actually train against. The asserts are the internal-consistency checks Lab 02
established: mass covered must rise with k, the renormalised estimate must sit at or above
the dense KL and the tail-bucket estimate at or below it, and both must tighten as k grows.
The storage line uses `bytes_per_token_cache` to complete the accounting.

In [5]:
from transformers import AutoModelForCausalLM

teacher = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-360M-Instruct", dtype=torch.float32).eval()
student = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M-Instruct", dtype=torch.float32).eval()
ids, mask = ev["input_ids"][:2, :192], ev["mask"][:2, :192]
with torch.no_grad():
    t_lg = teacher(ids).logits[:, :-1]
    s_lg = student(ids).logits[:, :-1]
mk = mask[:, 1:]
del teacher, student

rows = topk_truncation_bias(s_lg, t_lg, mk, ks=(1, 5, 64))
dense = rows[0]["dense_kl"]
V = t_lg.shape[-1]
print(f"dense forward KL on this pair: {dense:.4f} nats | vocab {V:,}")
print(f"{'k':>4} {'mass kept':>10} {'renorm KL':>10} {'tail KL':>9} {'GB/1M tokens':>13}")
for r in rows:
    gb = bytes_per_token_cache(V, k=r["k"])["topk_gb_per_1M_tokens"]
    print(f"{r['k']:>4} {r['mean_mass_covered']:>10.4f} {r['renorm_kl']:>10.4f} "
          f"{r['tail_bucket_kl']:>9.4f} {gb:>13.3f}")

masses = [r["mean_mass_covered"] for r in rows]
assert masses[0] < masses[1] < masses[2], "mass kept must rise with k"
assert all(r["renorm_kl"] >= dense - 1e-6 for r in rows), "renorm overstates KL (lab 02)"
assert all(r["tail_bucket_kl"] <= dense + 1e-6 for r in rows), "tail understates KL (lab 02)"
assert abs(rows[0]["renorm_rel_err"]) > abs(rows[-1]["renorm_rel_err"]), \
    "the distortion the defense adds must shrink as k grows"
withheld = [1 - m for m in masses]
print(f"\nmass withheld by the defense: k=1 {withheld[0]:.1%} | k=5 {withheld[1]:.1%} | "
      f"k=64 {withheld[2]:.1%}")

if RUN_EVAL:
    # The gated pricing runs: rebuild Lab 04's cache twice (TopKCacheWriter with
    # k=1 and k=5, same corpus fingerprint discipline), retrain the stage-2
    # student against each, and score both on the decontaminated harness. The
    # trade-off chart plots (defense k) x (student quality, exercise 3's axis).
    pass
else:
    print("RUN_EVAL=False: the retraining pair is gated; the price prediction stands above")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dense forward KL on this pair: 0.2939 nats | vocab 49,152
   k  mass kept  renorm KL   tail KL  GB/1M tokens
   1     0.7342     0.6543    0.1242         0.008
   5     0.9308     0.3674    0.2242         0.032
  64     0.9916     0.3040    0.2810         0.386

mass withheld by the defense: k=1 26.6% | k=5 6.9% | k=64 0.8%
RUN_EVAL=False: the retraining pair is gated; the price prediction stands above


**Interpretation.** The live table prices the defense honestly, and the price is
lower than a defender would hope. Even at k = 1, serving only the single top token's
probability, the teacher hands over 73% of its probability mass on the average position,
because Lab 02's finding holds here too: real instruct-model distributions are peaked. So
the harshest truncation available withholds only about a quarter of the distributional
information, and by k = 5 the withheld share is already down to 7%, as the printed
withheld-mass line shows. The bias columns bracket the damage to
the training signal: at k = 1 the two estimators disagree with the dense KL by the largest
margins in the table (the renormalised estimate above it, the tail-bucket estimate below,
exactly the opposite-direction pair Lab 02 measured), and both collapse toward dense by
k = 64, which the final assert confirms.

The prediction for the gated retraining, stated so the runs can refute it: the k = 1 student
should land near Lab 06's SeqKD arm (top-1 information is approximately "which token won",
which is what sampled text already reveals), the k = 5 student should recover most of the
k = 64 student's quality, and the whole quality spread should be modest, mirroring the modest
spread in the mass columns. Combined with exercise 3's finding, the defender's trade-off
chart reads bleakly from the defender's chair: truncation costs the attacker little quality,
and rate limits bind after the cheap part of the curve. What the two exercises jointly
recommend is defenses priced elsewhere: watermarking (which survives truncation because it
lives in the sampled text) and behavioral monitoring of query patterns, with logit truncation
kept mainly because it is nearly free to serve, not because it defends much.